In [1]:
# library imports
import pandas as pd
import re

## Data Cleaning

In [2]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/giovani-gutierrez/animal_shelter_outcomes/refs/heads/main/data/raw.csv"
)

In [3]:
# set all column names to be in snake case
for col in df.columns:
    new_name = re.sub("([a-z])([A-Z])", r"\1_\2", col)
    new_name = new_name.replace(" ", "_").lower()
    df.rename(columns={col: new_name}, inplace=True)

# preview new column names
df.columns

Index(['facility', 'animal_id', 'animal_type', 'animal_breed', 'impound_no',
       'admission_fy', 'intake_type', 'intake_group', 'outcome_fy',
       'outcome_type', 'outcome_group', 'intake_date', 'outcome_date',
       'object_id'],
      dtype='str')

In [4]:
# drop unnecessary columns
cols_to_drop = [
    "facility",
    "animal_id",
    "impound_no",
    "admission_fy",
    "outcome_fy",
    "outcome_type",
    "outcome_date",
    "object_id",
]

df = df.drop(columns=cols_to_drop)

In [5]:
# keep only those animals that were alive on intake
df = df[df["intake_group"] != "DECEASED"]
df = df.drop(columns=["intake_group"])

In [6]:
# keep only valid outcome groups
valid_outcomes = ["ADOPTION", "OTHER LIVE", "RESCUE", "RTO", "DIED", "EUTHANASIA"]

df = df[df["outcome_group"].isin(valid_outcomes)]

In [7]:
# map observation to target value
def outcome_map(x):
    if isinstance(x, str):
        x = x.upper()
        if x in ["ADOPTION", "OTHER LIVE", "RESCUE", "RTO"]:
            return "LIVE"
        elif x in ["DIED", "EUTHANASIA"]:
            return "NONLIVE"


df["outcome"] = df["outcome_group"].apply(outcome_map)
df = df.drop(columns=["outcome_group"])

# Feature Engineering & Further Data Cleaning

In [13]:
# create intake month and season columns
df["intake_date"] = pd.to_datetime(df["intake_date"])
df["intake_day"] = df["intake_date"].dt.day_name().str.upper()
df["intake_month"] = df["intake_date"].dt.month_name().str.upper()

month2season = {
    "JANUARY": "WINTER",
    "FEBRUARY": "WINTER",
    "MARCH": "SPRING",
    "APRIL": "SPRING",
    "MAY": "SPRING",
    "JUNE": "SUMMER",
    "JULY": "SUMMER",
    "AUGUST": "SUMMER",
    "SEPTEMBER": "FALL",
    "OCTOBER": "FALL",
    "NOVEMBER": "FALL",
    "DECEMBER": "WINTER",
}

df["intake_season"] = df["intake_month"].map(month2season)

In [17]:
# create boolean indicator for weekend intake
df["intake_is_weekend"] = df["intake_day"].isin(["SATURDAY", "SUNDAY"])

In [19]:
# drop intake date
df = df.drop(columns=["intake_date"]).reset_index(drop=True)

In [20]:
df.head()

,animal_type,animal_breed,intake_type,outcome,intake_day,intake_month,intake_season,intake_is_weekend
0,CATS,DOMESTIC LH,STRAY,LIVE,MONDAY,JUNE,SUMMER,False
1,DOGS,LABRADOR RETR,STRAY,LIVE,MONDAY,NOVEMBER,FALL,False
2,DOGS,LABRADOR RETR,STRAY,LIVE,SUNDAY,FEBRUARY,WINTER,True
3,DOGS,LABRADOR RETR,STRAY,LIVE,WEDNESDAY,APRIL,SPRING,False
4,DOGS,MALTESE,OWNER SUR,NONLIVE,WEDNESDAY,NOVEMBER,FALL,False


In [21]:
# save to csv
df.to_csv(
    "C:/Users/giova/Desktop/PersonalProjects/animal_shelter_outcomes/data/clean.csv",
    index=False,
)